In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [ ]:
# dataloader_binary.py 먼저 만들기
# model_binary_resnet.py로 모델 구조
# train_binary.py 작성해서 학습
# 성능 확인 후 → 필요 시 증강 적용 또는 ResNet34/Dropout 확장
# 사용 모델 RetNet18 모델

In [ ]:
# RetNet18 모델 구현
"""
torchvision.models.resnet18(pretrained=True) 로 사전학습 모델 사용
- ImageNet에 미리 학습된 가중치(weigth) 사용 : 1000개 클래스, 120만장 이미지 분류 데이터셋
  (전이학습)
마지막 fc 층을 클래스 수 2개 (food, not_food)에 맞게 교체
깔끔하게 함수로 정의해서 학습 스크립트에서 불러와 쓰기 좋게 구성
"""

In [2]:
import torch.nn as nn
from torchvision import models

In [3]:
def get_resnet18_binary_model(pretrained=True):
    """
    ResNet18 기반 이진 분류 모델 생성 함수
    :param pretrained: ImageNet 사전학습 가중치 사용할지 여부
    :return: nn.Module (food vs not_food 분류기)
    nn :  PyTorch의 신경망 모듈 / nn.ReLU()활성화 함수 
    nn.CrossEntropyLoss() : 분류 문제에서 자주 쓰는 손실 함수
    """
    # 모델 불러오기(ImageNet 사전학습 가중치 포함)
    model = models.resnet18(pretrained=pretrained)
    
    #  기존 fc layer의 입력 feature 수 확인
    # ImageNet 1000개 클래스를 예측하므로 출력층이 nn.Linear(512,1000)으로 되어있어
    # 2-class로 바꿔야 하니까 입력 크기 512를 꺼내서 씀
    num_ftrs = model.fc.in_features
    
    # fc layer  교체 : 출력 클래스 수 = 2(food, not_food)
    # linear(512,1000) ->lInear(512,2)로 바꿔서 food,not_food 분류되도록 만듬
    model.fc = nn.Linear(num_ftrs,2)
    
    return model